# Audio Transcription Diagnostics

Use this notebook to visually inspect already-generated `NoteEvent` JSON from the audio transcription path before tuning filters or decoder behavior.

This notebook does **not** run Basic Pitch and does **not** require real audio files. It only reads a user-provided JSON file containing a list of project `NoteEvent` records, such as output from:

```bash
tabgen audio-to-notes input.wav --out notes.json
```

Do not commit user recordings, generated `notes.json`, filtered notes, or generated plots to the repository.

Raw Basic Pitch output may include harmonics, low-frequency artifacts, short spurious notes, and low-confidence notes. The controls below are simple diagnostics, not production filter defaults.

## Optional Local Packages

`pandas` and `matplotlib` are intentionally not project runtime dependencies. Install them only in a local diagnostics environment if needed, for example:

```bash
uv add --dev pandas matplotlib
```

Avoid committing dependency changes unless a dedicated issue asks for notebook/dev-tool dependencies.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")

## Configure Input and Filters

Set `notes_json_path` to a local `NoteEvent` JSON file. The file should contain a JSON list with fields like `start`, `end`, `pitch_midi`, `confidence`, and `source`.

In [ ]:
notes_json_path = Path("notes.json")

min_confidence = 0.0
min_duration = 0.0
min_pitch = 0
max_pitch = 127

save_filtered_notes = False
filtered_notes_path = Path("filtered_notes.json")

## Load NoteEvent JSON

In [ ]:
if not notes_json_path.exists():
    raise FileNotFoundError(
        f"Set notes_json_path to an existing NoteEvent JSON file: {notes_json_path}"
    )

notes_df = pd.read_json(notes_json_path)
required_columns = {"start", "end", "pitch_midi", "confidence"}
missing_columns = required_columns - set(notes_df.columns)
if missing_columns:
    raise ValueError(f"Missing required NoteEvent columns: {sorted(missing_columns)}")

notes_df = notes_df.copy()
notes_df["duration"] = notes_df["end"] - notes_df["start"]
notes_df = notes_df.sort_values(["start", "end", "pitch_midi"]).reset_index(drop=True)
notes_df.head()

## Summary Statistics

In [ ]:
if notes_df.empty:
    summary = pd.DataFrame(
        [
            {
                "note_count": 0,
                "pitch_range": None,
                "confidence_range": None,
                "duration_range": None,
                "total_time_span": 0.0,
            }
        ]
    )
else:
    summary = pd.DataFrame(
        [
            {
                "note_count": len(notes_df),
                "pitch_range": (notes_df["pitch_midi"].min(), notes_df["pitch_midi"].max()),
                "confidence_range": (notes_df["confidence"].min(), notes_df["confidence"].max()),
                "duration_range": (notes_df["duration"].min(), notes_df["duration"].max()),
                "total_time_span": notes_df["end"].max() - notes_df["start"].min(),
            }
        ]
    )

summary

## Raw Note Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].scatter(notes_df["start"], notes_df["pitch_midi"], s=16, alpha=0.75)
axes[0, 0].set_title("Pitch vs Start Time")
axes[0, 0].set_xlabel("Start time (s)")
axes[0, 0].set_ylabel("MIDI pitch")

axes[0, 1].scatter(notes_df["start"], notes_df["confidence"], s=16, alpha=0.75)
axes[0, 1].set_title("Confidence vs Start Time")
axes[0, 1].set_xlabel("Start time (s)")
axes[0, 1].set_ylabel("Confidence")

axes[1, 0].hist(notes_df["duration"], bins=40)
axes[1, 0].set_title("Note Duration Histogram")
axes[1, 0].set_xlabel("Duration (s)")
axes[1, 0].set_ylabel("Count")

axes[1, 1].hist(notes_df["pitch_midi"], bins=range(0, 129))
axes[1, 1].set_title("Pitch Histogram")
axes[1, 1].set_xlabel("MIDI pitch")
axes[1, 1].set_ylabel("Count")

fig.tight_layout()

## Apply Simple Diagnostic Filters

These filters help inspect likely noisy regions. They are not production defaults.

In [ ]:
filter_mask = (
    (notes_df["confidence"] >= min_confidence)
    & (notes_df["duration"] >= min_duration)
    & (notes_df["pitch_midi"] >= min_pitch)
    & (notes_df["pitch_midi"] <= max_pitch)
)

filtered_df = notes_df.loc[filter_mask].copy().reset_index(drop=True)

pd.DataFrame(
    [
        {
            "before_count": len(notes_df),
            "after_count": len(filtered_df),
            "removed_count": len(notes_df) - len(filtered_df),
            "min_confidence": min_confidence,
            "min_duration": min_duration,
            "min_pitch": min_pitch,
            "max_pitch": max_pitch,
        }
    ]
)

## Filtered Pitch Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(filtered_df["start"], filtered_df["pitch_midi"], s=18, alpha=0.8)
ax.set_title("Filtered Pitch vs Start Time")
ax.set_xlabel("Start time (s)")
ax.set_ylabel("MIDI pitch")
fig.tight_layout()

## Optionally Save Filtered Notes

This writes only when `save_filtered_notes = True`. Do not commit generated filtered notes unless a dedicated fixture issue asks for it.

In [ ]:
if save_filtered_notes:
    note_columns = [column for column in ["start", "end", "pitch_midi", "confidence", "source"] if column in filtered_df]
    filtered_df[note_columns].to_json(
        filtered_notes_path,
        orient="records",
        indent=2,
    )
    print(f"Saved filtered notes to {filtered_notes_path}")
else:
    print("Set save_filtered_notes = True to write filtered_notes.json locally.")